In [ ]:
from pathlib import Path
from urllib.parse import urlparse

import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist
from sklearn.cluster import AgglomerativeClustering, KMeans, MeanShift
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import StratifiedGroupKFold


In [2]:
if Path.cwd().name == "training":
    PROJECT_ROOT = Path.cwd().parent
else:
    PROJECT_ROOT = Path.cwd()

DATA_PATH = PROJECT_ROOT / "data" / "phishing_url_features.csv"

data = pd.read_csv(DATA_PATH)

X = data.drop(columns=["url", "label"])

y = data["label"].astype(int)

print("전체 데이터:", data.shape)
print()
print("Label 분포:")
print(y.value_counts())

전체 데이터: (22000, 17)

Label 분포:
label
0    15000
1     7000
Name: count, dtype: int64


In [3]:
def get_hostname(url):
    return (urlparse(str(url)).hostname or "").lower()


groups = data["url"].apply(get_hostname)

print("고유 hostname:", groups.nunique())

고유 hostname: 20184


In [4]:
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

train_idx, test_idx = next(sgkf.split(X, y, groups))


X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

train_groups = groups.iloc[train_idx]
test_groups = groups.iloc[test_idx]

In [5]:
print("Train:", X_train.shape)

print(y_train.value_counts())

print()

print("Test:", X_test.shape)

print(y_test.value_counts())

print()

overlap = set(train_groups) & set(test_groups)

print("Train/Test hostname 중복:", len(overlap))

Train: (17600, 15)
label
0    12000
1     5600
Name: count, dtype: int64

Test: (4400, 15)
label
0    3000
1    1400
Name: count, dtype: int64

Train/Test hostname 중복: 0


In [6]:
results = []


def evaluate_model(model_name, y_true, y_pred):
    accuracy = accuracy_score(y_true, y_pred)

    balanced_accuracy = balanced_accuracy_score(y_true, y_pred)

    precision = precision_score(y_true, y_pred, zero_division=0)

    recall = recall_score(y_true, y_pred, zero_division=0)

    f1 = f1_score(y_true, y_pred, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    result = {
        "Model": model_name,
        "Accuracy": accuracy,
        "Balanced Accuracy": balanced_accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
    }

    results.append(result)

    return result

In [7]:
def create_cluster_mapping(cluster_labels, y_train):
    mapping_df = pd.DataFrame({"cluster": cluster_labels, "label": y_train.to_numpy()})

    cluster_mean = mapping_df.groupby("cluster")["label"].mean()

    mapping = {
        cluster: (1 if mean >= 0.5 else 0) for cluster, mean in cluster_mean.items()
    }

    return mapping

In [8]:
kmeans = KMeans(n_clusters=2, n_init=10, random_state=42)

kmeans.fit(X_train)


cluster_mapping = create_cluster_mapping(kmeans.labels_, y_train)

print("Cluster mapping:", cluster_mapping)


test_clusters = kmeans.predict(X_test)


y_pred_kmeans = np.array([cluster_mapping[cluster] for cluster in test_clusters])


evaluate_model("K-Means", y_test, y_pred_kmeans)

Cluster mapping: {0: 0, 1: 1}


{'Model': 'K-Means',
 'Accuracy': 0.9172727272727272,
 'Balanced Accuracy': 0.9368571428571428,
 'Precision': 0.7980437284234753,
 'Recall': 0.9907142857142858,
 'F1': 0.8840025493945188,
 'TN': np.int64(2649),
 'FP': np.int64(351),
 'FN': np.int64(13),
 'TP': np.int64(1387)}

In [9]:
gmm = GaussianMixture(n_components=2, n_init=10, random_state=42)

gmm.fit(X_train)


train_clusters = gmm.predict(X_train)


cluster_mapping = create_cluster_mapping(train_clusters, y_train)


test_clusters = gmm.predict(X_test)


y_pred_gmm = np.array([cluster_mapping[cluster] for cluster in test_clusters])


evaluate_model("GMM", y_test, y_pred_gmm)

{'Model': 'GMM',
 'Accuracy': 0.9163636363636364,
 'Balanced Accuracy': 0.9359999999999999,
 'Precision': 0.7965517241379311,
 'Recall': 0.99,
 'F1': 0.8828025477707007,
 'TN': np.int64(2646),
 'FP': np.int64(354),
 'FN': np.int64(14),
 'TP': np.int64(1386)}

In [10]:
mean_shift_seeds = np.unique(X_train.to_numpy(), axis=0)

print("MeanShift seed 개수:", len(mean_shift_seeds))

MeanShift seed 개수: 541


In [11]:
meanshift = MeanShift(bandwidth=2.27, seeds=mean_shift_seeds)

meanshift.fit(X_train)


cluster_mapping = create_cluster_mapping(meanshift.labels_, y_train)


test_clusters = meanshift.predict(X_test)


y_pred_meanshift = np.array([cluster_mapping[cluster] for cluster in test_clusters])


evaluate_model("MeanShift", y_test, y_pred_meanshift)

{'Model': 'MeanShift',
 'Accuracy': 0.8293181818181818,
 'Balanced Accuracy': 0.7550238095238095,
 'Precision': 0.8633818589025756,
 'Recall': 0.5507142857142857,
 'F1': 0.672481465329263,
 'TN': np.int64(2878),
 'FP': np.int64(122),
 'FN': np.int64(629),
 'TP': np.int64(771)}

In [12]:
random_forest = RandomForestClassifier(n_estimators=100, random_state=42)


random_forest.fit(X_train, y_train)


y_pred_rf = random_forest.predict(X_test)


evaluate_model("Random Forest", y_test, y_pred_rf)

{'Model': 'Random Forest',
 'Accuracy': 0.9681818181818181,
 'Balanced Accuracy': 0.9644761904761905,
 'Precision': 0.9461756373937678,
 'Recall': 0.9542857142857143,
 'F1': 0.9502133712660028,
 'TN': np.int64(2924),
 'FP': np.int64(76),
 'FN': np.int64(64),
 'TP': np.int64(1336)}

In [13]:
logistic_regression = LogisticRegression(random_state=42, max_iter=1000)


logistic_regression.fit(X_train, y_train)


y_pred_lr = logistic_regression.predict(X_test)


evaluate_model("Logistic Regression", y_test, y_pred_lr)

{'Model': 'Logistic Regression',
 'Accuracy': 0.9622727272727273,
 'Balanced Accuracy': 0.9641428571428572,
 'Precision': 0.9168918918918919,
 'Recall': 0.9692857142857143,
 'F1': 0.9423611111111111,
 'TN': np.int64(2877),
 'FP': np.int64(123),
 'FN': np.int64(43),
 'TP': np.int64(1357)}

In [14]:
results_df = pd.DataFrame(results)

metric_columns = ["Accuracy", "Balanced Accuracy", "Precision", "Recall", "F1"]

results_df[metric_columns] = results_df[metric_columns] * 100


display(results_df.round(2))

,Model,Accuracy,Balanced Accuracy,Precision,Recall,F1,TN,FP,FN,TP
0,K-Means,91.73,93.69,79.80,99.07,88.40,2649,351,13,1387
1,GMM,91.64,93.60,79.66,99.00,88.28,2646,354,14,1386
2,MeanShift,82.93,75.50,86.34,55.07,67.25,2878,122,629,771
3,Random Forest,96.82,96.45,94.62,95.43,95.02,2924,76,64,1336
4,Logistic Regression,96.23,96.41,91.69,96.93,94.24,2877,123,43,1357


In [15]:
agglomerative = AgglomerativeClustering(n_clusters=2)

train_clusters = agglomerative.fit_predict(X_train)

print("Train cluster distribution:", np.unique(train_clusters, return_counts=True))

Train cluster distribution: (array([0, 1]), array([9985, 7615]))


In [16]:
agglomerative_mapping = create_cluster_mapping(train_clusters, y_train)

print("Cluster mapping:", agglomerative_mapping)

Cluster mapping: {0: 0, 1: 1}


In [17]:
def calculate_cluster_centers(X_data, cluster_labels):
    X_array = np.asarray(X_data)

    cluster_labels = np.asarray(cluster_labels)

    centers = {}

    for cluster in np.unique(cluster_labels):
        cluster_points = X_array[cluster_labels == cluster]

        centers[cluster] = cluster_points.mean(axis=0)

    return centers


agglomerative_centers = calculate_cluster_centers(X_train, train_clusters)

for cluster, center in agglomerative_centers.items():
    print(f"Cluster {cluster}:", center)

Cluster 0: [1.         1.         1.         1.         1.         1.
 0.99679519 0.48472709 0.23104657 1.         1.         0.41802704
 1.         0.90415623 0.98858287]
Cluster 1: [ 0.99369665  0.79343401  0.93381484  0.98292843  0.99369665  0.38305975
  0.68207485  0.09494419 -0.25594222 -0.8219304   0.99973736  0.46999343
  0.8093237   0.75272489  0.78791858]


In [18]:
def predict_nearest_cluster(X_test, cluster_centers):
    center_labels = list(cluster_centers.keys())

    centers = np.array([cluster_centers[label] for label in center_labels])

    distances = cdist(X_test, centers, metric="euclidean")

    nearest_indices = np.argmin(distances, axis=1)

    predicted_clusters = np.array([center_labels[index] for index in nearest_indices])

    return predicted_clusters


test_clusters = predict_nearest_cluster(X_test.to_numpy(), agglomerative_centers)

print("Test cluster distribution:", np.unique(test_clusters, return_counts=True))

Test cluster distribution: (array([0, 1]), array([2654, 1746]))


In [19]:
y_pred_agglomerative = np.array(
    [agglomerative_mapping[cluster] for cluster in test_clusters]
)

In [20]:
evaluate_model("Agglomerative", y_test, y_pred_agglomerative)

{'Model': 'Agglomerative',
 'Accuracy': 0.9154545454545454,
 'Balanced Accuracy': 0.9355238095238095,
 'Precision': 0.7943871706758304,
 'Recall': 0.9907142857142858,
 'F1': 0.8817546090273363,
 'TN': np.int64(2641),
 'FP': np.int64(359),
 'FN': np.int64(13),
 'TP': np.int64(1387)}

In [21]:
results_df = pd.DataFrame(results)

metric_columns = ["Accuracy", "Balanced Accuracy", "Precision", "Recall", "F1"]

results_df[metric_columns] = results_df[metric_columns] * 100

display(results_df.round(2))

,Model,Accuracy,Balanced Accuracy,Precision,Recall,F1,TN,FP,FN,TP
0,K-Means,91.73,93.69,79.80,99.07,88.40,2649,351,13,1387
1,GMM,91.64,93.60,79.66,99.00,88.28,2646,354,14,1386
2,MeanShift,82.93,75.50,86.34,55.07,67.25,2878,122,629,771
3,Random Forest,96.82,96.45,94.62,95.43,95.02,2924,76,64,1336
4,Logistic Regression,96.23,96.41,91.69,96.93,94.24,2877,123,43,1357
5,Agglomerative,91.55,93.55,79.44,99.07,88.18,2641,359,13,1387
